## Checking GPU and ENV

In [28]:
!nvidia-smi

Fri Mar 20 08:34:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.211.01             Driver Version: 570.211.01     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   36C    P8             14W /  350W |      88MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Installs and Imports

In [30]:
!pip install requests tqdm chromadb sentence-transformers ijson -q

## Setting Up Vector Store

In [31]:
import requests
import json
import os
from tqdm import tqdm
import chromadb

# --- Configuration ---
DATASET_URL = "https://zenodo.org/records/15808027/files/arxiv.json?download=1"
FILE_NAME = "arxiv.json"
DB_PATH = "./arxiv_vectordb"
BATCH_SIZE = 500

# 1. Download the Dataset
if not os.path.exists(FILE_NAME):
    print("Downloading 4.7 GB dataset... This might take a while.")
    response = requests.get(DATASET_URL, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024 * 1024  # 1 MB
    
    with open(FILE_NAME, 'wb') as file, tqdm(total=total_size, unit='iB', unit_scale=True, desc="Downloading") as bar:
        for data in response.iter_content(block_size):
            bar.update(len(data))
            file.write(data)
    print("Download complete!")
else:
    print(f"Dataset '{FILE_NAME}' already exists locally.")


Dataset 'arxiv.json' already exists locally.


In [ ]:
import chromadb

# 1. Connect to your existing database
client = chromadb.PersistentClient(path="./arxiv_vectordb")

# 2. Delete the collection
try:
    client.delete_collection(name="arxiv_papers")
    print("Success: The 'arxiv_papers' collection and all its vectors have been deleted.")
except Exception as e:
    print(f"Notice: Could not delete collection (it might already be empty or not exist). Error: {e}")


Success: The 'arxiv_papers' collection and all its vectors have been deleted.


In [34]:
import ijson
import os
from tqdm import tqdm
import chromadb

FILE_NAME = "arxiv.json"
DB_PATH = "./arxiv_vectordb"
BATCH_SIZE = 500
MAX_DOCS_TO_PROCESS = None 

print("Initializing ChromaDB...")
client = chromadb.PersistentClient(path=DB_PATH)
collection = client.get_or_create_collection(name="arxiv_papers")

documents = []
metadatas = []
ids = []

print("Streaming JSON and adding to Vector DB...")

# ijson works best when opening the file in binary read mode ('rb')
with open(FILE_NAME, 'rb') as f:
    
    # This specifically looks for items inside the "edges" array
    parser = ijson.items(f, 'edges.item')
    
    for i, edge in enumerate(tqdm(parser, desc="Processing papers")):
        
        if MAX_DOCS_TO_PROCESS and i >= MAX_DOCS_TO_PROCESS:
            print(f"\nStopping at {MAX_DOCS_TO_PROCESS} records as configured.")
            break
            
        # The paper data is inside the 'attrs' key based on your snippet
        attrs = edge.get('attrs', {})
        
        # Extract fields (using defaults if they happen to be missing)
        title = attrs.get('title', 'No Title')
        
        # The dataset might not have an 'abstract' in this specific network view,
        # but we will try to grab it, or fall back to comments/categories.
        abstract = attrs.get('abstract', 'No Abstract provided in dataset')
        
        text_to_embed = f"Title: {title}\nAbstract: {abstract}"
        
        # Safely extract and stringify ALL metadata values to appease ChromaDB
        raw_categories = attrs.get('categories')
        if isinstance(raw_categories, list):
            categories_str = ", ".join([str(c) for c in raw_categories if c])
        else:
            categories_str = str(raw_categories or 'Unknown')
            
        metadata = {
            "submitter": str(attrs.get('submitter') or 'Unknown'),
            "categories": categories_str,
            "comments": str(attrs.get('comments') or 'None')
        }
        
        # Generate an ID (using the paper title or loop index since 'id' isn't visible in the snippet)
        doc_id = attrs.get('id', f"doc_{i}")
        
        documents.append(text_to_embed)
        metadatas.append(metadata)
        ids.append(doc_id)
        
        # Insert into Vector DB in batches
        if len(documents) >= BATCH_SIZE:
            collection.add(
                documents=documents,
                metadatas=metadatas,
                ids=ids
            )
            documents, metadatas, ids = [], [], []

# Insert any final remaining documents
if documents:
    collection.add(documents=documents, metadatas=metadatas, ids=ids)

print(f"\nSuccess! Total documents in vector collection: {collection.count()}")

Initializing ChromaDB...
Streaming JSON and adding to Vector DB...


Processing papers: 1282999it [3:50:59, 92.57it/s]


InternalError: Error in compaction: Failed to apply logs to the metadata segment

## Testing Retreival

In [32]:
import chromadb

# 1. Re-connect to your existing database
# (Use get_collection instead of get_or_create to ensure it exists)
client = chromadb.PersistentClient(path="./arxiv_vectordb")
collection = client.get_collection(name="arxiv_papers")

# 2. Define your search
search_query = "efficient state-space models and neural networks"
print(f"Searching for: '{search_query}'...\n")

# 3. Execute the query
results = collection.query(
    query_texts=[search_query],
    n_results=3, # Change this to get more or fewer results
    
    # ADVANCED: You can uncomment the 'where' clause to filter by the metadata we cleaned!
    # For example, to only search for papers that contain "cs.LG" (Machine Learning) in their categories:
    # where={"categories": {"$contains": "cs.LG"}}
)

# 4. Parse and display the output
# ChromaDB returns lists of lists (to support multiple queries at once), so we index at [0]
documents = results['documents'][0]
metadatas = results['metadatas'][0]
distances = results['distances'][0] 

for i in range(len(documents)):
    print(f"--- Result {i+1} ---")
    # Distance measures how "close" the vectors are. Lower distance = higher relevance.
    print(f"Relevance Score (Distance): {distances[i]:.4f}\n") 
    
    print(f"Content:\n{documents[i]}\n")
    
    print("Metadata:")
    for key, value in metadatas[i].items():
        print(f"  - {key}: {value}")
    print("\n" + "="*50 + "\n")

Searching for: 'efficient state-space models and neural networks'...

--- Result 1 ---
Relevance Score (Distance): 0.9372

Content:
Title: Control flow in active inference systems
Abstract:   Living systems face both environmental complexity and limited access to
free-energy resources. Survival under these conditions requires a control
system that can activate, or deploy, available perception and action resources
in a context specific way. We show here that when systems are described as
executing active inference driven by the free-energy principle (and hence can
be considered Bayesian prediction-error minimizers), their control flow systems
can always be represented as tensor networks (TNs). We show how TNs as control
systems can be implmented within the general framework of quantum topological
neural networks, and discuss the implications of these results for modeling
biological systems at multiple scales.


Metadata:
  - comments: 44 pgs
  - categories: q-bio.NC, physics.bio-ph, qua